# Lab 20 — Multi-Agent Research Demo Notebook

Notebook này giúp bạn **thử nghiệm nhanh** các khối logic của bài lab trước khi implement chính thức trong `src/`.

**Luồng làm việc:**
1. Khám phá schemas & shared state
2. Mock services (LLM + Search) để chạy không cần API key
3. Viết các agent demo (Researcher → Analyst → Writer)
4. Supervisor routing + vòng lặp workflow mini
5. Benchmark single-agent vs multi-agent

> ⚠️ **Quy tắc:** Notebook chỉ để prototype. Sau khi chạy được ở đây, bạn phải **chuyển logic vào `src/multi_agent_research_lab/`** và pass tests. Các ô có `TODO(student)` là phần bạn phải tự viết.

## 0. Setup

Chạy từ repo root với package đã cài (`pip install -e ".[dev]"`).

In [ ]:
import sys
from pathlib import Path

# Cho phép import package khi chạy notebook từ thư mục notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

from multi_agent_research_lab.core.errors import StudentTodoError
from multi_agent_research_lab.core.schemas import (
    AgentName,
    AgentResult,
    BenchmarkMetrics,
    ResearchQuery,
    SourceDocument,
)
from multi_agent_research_lab.core.state import ResearchState

print("✅ Import OK — package sẵn sàng")

## 1. Khám phá Shared State

`ResearchState` là **single source of truth** được truyền qua mọi agent. Mỗi agent đọc state, cập nhật, rồi trả lại.

In [ ]:
query = ResearchQuery(
    query="So sánh RAG và fine-tuning cho domain adaptation",
    max_sources=3,
)
state = ResearchState(request=query)

state.record_route("researcher")
state.add_trace_event("demo", {"note": "first route recorded"})

print("Iteration:", state.iteration)
print("Route history:", state.route_history)
print("Trace:", state.trace)

## 2. Mock Services

Để demo không cần API key, ta dùng mock. Trong bản chính thức (`src/services/`), bạn sẽ nối provider thật (OpenAI / Tavily...).

- `MockSearchClient`: **đã viết sẵn** làm mẫu.
- `MockLLMClient`: **TODO(student)** — bạn tự viết theo cùng pattern.

In [ ]:
from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        sys_lower = system_prompt.lower()
        in_tokens = max(10, len(system_prompt + user_prompt) // 4)
        if "analyst" in sys_lower:
            content = (
                "### Phân tích nguồn và So sánh kỹ thuật\n"
                "1. RAG tối ưu cho tri thức động và cần truy xuất chính xác kèm trích dẫn.\n"
                "2. Fine-tuning tối ưu cho phong cách phản hồi chuyên biệt và giảm độ trễ trên tập dữ liệu tĩnh.\n"
                "3. Đánh giá nguồn: Cả 3 nguồn đều thống nhất mô hình hybrid (RAG + Fine-tuning) mang lại kết quả cao nhất."
            )
        elif "writer" in sys_lower:
            content = (
                "# So sánh RAG và Fine-tuning cho Domain Adaptation\n\n"
                "## Tóm tắt tổng quan\n"
                "Trong việc thích ứng miền dữ liệu cho LLM, RAG và Fine-tuning phục vụ các mục tiêu bổ trợ lẫn nhau [1].\n\n"
                "## So sánh chi tiết\n"
                "- **RAG (Retrieval-Augmented Generation):** Giảm thiểu hallucination bằng cách truy xuất tài liệu ngoài theo thời gian thực [2].\n"
                "- **Fine-tuning:** Hiệu quả khi cần hành vi nhất quán, kiểm soát định dạng, và tối ưu hóa độ trễ [3].\n\n"
                "## Khuyến nghị\n"
                "Nên áp dụng RAG cho kho tài liệu thường xuyên cập nhật và kết hợp Fine-tuning khi cần học thuật ngữ hoặc phong cách riêng.\n\n"
                "## References\n"
                "[1] RAG vs Fine-tuning: A Practical Guide (https://example.com/rag-vs-ft)\n"
                "[2] Retrieval-Augmented Generation Survey (https://example.com/rag-survey)\n"
                "[3] When to Fine-tune LLMs (https://example.com/when-finetune)"
            )
        else:
            content = (
                "RAG và Fine-tuning là hai phương pháp chính để domain adaptation. "
                "RAG truy xuất ngữ cảnh mới từ cơ sở dữ liệu ngoài, trong khi Fine-tuning cập nhật trọng số mô hình."
            )
        out_tokens = max(15, len(content) // 4)
        return MockLLMResponse(content=content, input_tokens=in_tokens, output_tokens=out_tokens)


# Smoke test phần đã cho sẵn
search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")


## 3. Demo Agents

Mỗi agent tuân theo contract `BaseAgent.run(state) -> state`.

- `DemoResearcherAgent`: **đã viết sẵn** làm mẫu — gọi search, ghi `sources` + `research_notes`.
- `DemoAnalystAgent`: **TODO(student)** — tổng hợp `sources` thành `analysis_notes`.
- `DemoWriterAgent`: **TODO(student)** — viết `final_answer` kèm citation.

In [ ]:
from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        sys_lower = system_prompt.lower()
        in_tokens = max(10, len(system_prompt + user_prompt) // 4)
        if "analyst" in sys_lower:
            content = (
                "### Phân tích nguồn và So sánh kỹ thuật\n"
                "1. RAG tối ưu cho tri thức động và cần truy xuất chính xác kèm trích dẫn.\n"
                "2. Fine-tuning tối ưu cho phong cách phản hồi chuyên biệt và giảm độ trễ trên tập dữ liệu tĩnh.\n"
                "3. Đánh giá nguồn: Cả 3 nguồn đều thống nhất mô hình hybrid (RAG + Fine-tuning) mang lại kết quả cao nhất."
            )
        elif "writer" in sys_lower:
            content = (
                "# So sánh RAG và Fine-tuning cho Domain Adaptation\n\n"
                "## Tóm tắt tổng quan\n"
                "Trong việc thích ứng miền dữ liệu cho LLM, RAG và Fine-tuning phục vụ các mục tiêu bổ trợ lẫn nhau [1].\n\n"
                "## So sánh chi tiết\n"
                "- **RAG (Retrieval-Augmented Generation):** Giảm thiểu hallucination bằng cách truy xuất tài liệu ngoài theo thời gian thực [2].\n"
                "- **Fine-tuning:** Hiệu quả khi cần hành vi nhất quán, kiểm soát định dạng, và tối ưu hóa độ trễ [3].\n\n"
                "## Khuyến nghị\n"
                "Nên áp dụng RAG cho kho tài liệu thường xuyên cập nhật và kết hợp Fine-tuning khi cần học thuật ngữ hoặc phong cách riêng.\n\n"
                "## References\n"
                "[1] RAG vs Fine-tuning: A Practical Guide (https://example.com/rag-vs-ft)\n"
                "[2] Retrieval-Augmented Generation Survey (https://example.com/rag-survey)\n"
                "[3] When to Fine-tune LLMs (https://example.com/when-finetune)"
            )
        else:
            content = (
                "RAG và Fine-tuning là hai phương pháp chính để domain adaptation. "
                "RAG truy xuất ngữ cảnh mới từ cơ sở dữ liệu ngoài, trong khi Fine-tuning cập nhật trọng số mô hình."
            )
        out_tokens = max(15, len(content) // 4)
        return MockLLMResponse(content=content, input_tokens=in_tokens, output_tokens=out_tokens)


# Smoke test phần đã cho sẵn
search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")


## 4. Supervisor Routing

Supervisor quyết định agent nào chạy tiếp dựa trên state hiện tại. Đây là **trái tim của bài lab** — bạn tự thiết kế policy.

In [ ]:
MAX_ITERATIONS = 6


def demo_supervisor_route(state: ResearchState) -> str:
    """Trả về một trong: 'researcher' | 'analyst' | 'writer' | 'done'."""
    # Guard chống vòng lặp vô hạn — GIỮ NGUYÊN dòng này
    if state.iteration >= MAX_ITERATIONS:
        return "done"

    if state.final_answer is not None:
        return "done"
    if not state.sources or not state.research_notes:
        return "researcher"
    if not state.analysis_notes:
        return "analyst"
    if not state.final_answer:
        return "writer"
    return "done"


## 5. Mini Workflow Loop

Vòng lặp điều phối **đã viết sẵn** — chỉ chạy được sau khi bạn hoàn thành các TODO ở trên. Đây chính là logic bạn sẽ chuyển thành LangGraph nodes/edges trong `graph/workflow.py`.

In [ ]:
from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        sys_lower = system_prompt.lower()
        in_tokens = max(10, len(system_prompt + user_prompt) // 4)
        if "analyst" in sys_lower:
            content = (
                "### Phân tích nguồn và So sánh kỹ thuật\n"
                "1. RAG tối ưu cho tri thức động và cần truy xuất chính xác kèm trích dẫn.\n"
                "2. Fine-tuning tối ưu cho phong cách phản hồi chuyên biệt và giảm độ trễ trên tập dữ liệu tĩnh.\n"
                "3. Đánh giá nguồn: Cả 3 nguồn đều thống nhất mô hình hybrid (RAG + Fine-tuning) mang lại kết quả cao nhất."
            )
        elif "writer" in sys_lower:
            content = (
                "# So sánh RAG và Fine-tuning cho Domain Adaptation\n\n"
                "## Tóm tắt tổng quan\n"
                "Trong việc thích ứng miền dữ liệu cho LLM, RAG và Fine-tuning phục vụ các mục tiêu bổ trợ lẫn nhau [1].\n\n"
                "## So sánh chi tiết\n"
                "- **RAG (Retrieval-Augmented Generation):** Giảm thiểu hallucination bằng cách truy xuất tài liệu ngoài theo thời gian thực [2].\n"
                "- **Fine-tuning:** Hiệu quả khi cần hành vi nhất quán, kiểm soát định dạng, và tối ưu hóa độ trễ [3].\n\n"
                "## Khuyến nghị\n"
                "Nên áp dụng RAG cho kho tài liệu thường xuyên cập nhật và kết hợp Fine-tuning khi cần học thuật ngữ hoặc phong cách riêng.\n\n"
                "## References\n"
                "[1] RAG vs Fine-tuning: A Practical Guide (https://example.com/rag-vs-ft)\n"
                "[2] Retrieval-Augmented Generation Survey (https://example.com/rag-survey)\n"
                "[3] When to Fine-tune LLMs (https://example.com/when-finetune)"
            )
        else:
            content = (
                "RAG và Fine-tuning là hai phương pháp chính để domain adaptation. "
                "RAG truy xuất ngữ cảnh mới từ cơ sở dữ liệu ngoài, trong khi Fine-tuning cập nhật trọng số mô hình."
            )
        out_tokens = max(15, len(content) // 4)
        return MockLLMResponse(content=content, input_tokens=in_tokens, output_tokens=out_tokens)


# Smoke test phần đã cho sẵn
search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")


## 6. Benchmark: Single-agent vs Multi-agent

Dùng `run_benchmark` từ package để so sánh. Baseline single-agent (1 lần gọi LLM, không search) **bạn tự viết**.

In [ ]:
from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        sys_lower = system_prompt.lower()
        in_tokens = max(10, len(system_prompt + user_prompt) // 4)
        if "analyst" in sys_lower:
            content = (
                "### Phân tích nguồn và So sánh kỹ thuật\n"
                "1. RAG tối ưu cho tri thức động và cần truy xuất chính xác kèm trích dẫn.\n"
                "2. Fine-tuning tối ưu cho phong cách phản hồi chuyên biệt và giảm độ trễ trên tập dữ liệu tĩnh.\n"
                "3. Đánh giá nguồn: Cả 3 nguồn đều thống nhất mô hình hybrid (RAG + Fine-tuning) mang lại kết quả cao nhất."
            )
        elif "writer" in sys_lower:
            content = (
                "# So sánh RAG và Fine-tuning cho Domain Adaptation\n\n"
                "## Tóm tắt tổng quan\n"
                "Trong việc thích ứng miền dữ liệu cho LLM, RAG và Fine-tuning phục vụ các mục tiêu bổ trợ lẫn nhau [1].\n\n"
                "## So sánh chi tiết\n"
                "- **RAG (Retrieval-Augmented Generation):** Giảm thiểu hallucination bằng cách truy xuất tài liệu ngoài theo thời gian thực [2].\n"
                "- **Fine-tuning:** Hiệu quả khi cần hành vi nhất quán, kiểm soát định dạng, và tối ưu hóa độ trễ [3].\n\n"
                "## Khuyến nghị\n"
                "Nên áp dụng RAG cho kho tài liệu thường xuyên cập nhật và kết hợp Fine-tuning khi cần học thuật ngữ hoặc phong cách riêng.\n\n"
                "## References\n"
                "[1] RAG vs Fine-tuning: A Practical Guide (https://example.com/rag-vs-ft)\n"
                "[2] Retrieval-Augmented Generation Survey (https://example.com/rag-survey)\n"
                "[3] When to Fine-tune LLMs (https://example.com/when-finetune)"
            )
        else:
            content = (
                "RAG và Fine-tuning là hai phương pháp chính để domain adaptation. "
                "RAG truy xuất ngữ cảnh mới từ cơ sở dữ liệu ngoài, trong khi Fine-tuning cập nhật trọng số mô hình."
            )
        out_tokens = max(15, len(content) // 4)
        return MockLLMResponse(content=content, input_tokens=in_tokens, output_tokens=out_tokens)


# Smoke test phần đã cho sẵn
search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")


## 7. Next Steps — chuyển sang `src/`

Khi notebook chạy end-to-end, chuyển logic vào code chính thức:

| Notebook | Đích trong `src/multi_agent_research_lab/` |
|---|---|
| `MockLLMClient` → provider thật | `services/llm_client.py` |
| `MockSearchClient` → provider thật | `services/search_client.py` |
| `DemoResearcherAgent` / `DemoAnalystAgent` / `DemoWriterAgent` | `agents/researcher.py`, `agents/analyst.py`, `agents/writer.py` |
| `demo_supervisor_route` | `agents/supervisor.py` |
| `run_demo_workflow` → LangGraph nodes/edges | `graph/workflow.py` |
| `compute_citation_coverage` + quality score | `evaluation/benchmark.py` |

Sau đó verify:
```bash
make lint && make test
python -m multi_agent_research_lab.cli run --query "..."
bash scripts/check_todos.sh   # đảm bảo không còn TODO trong src/
```